In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
anime = pd.read_csv("C:/Users/shanm/Python_Practice/Datasets/anime.csv")
anime.head(3)

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262


# Data Preprocessing:

In [3]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [4]:
# Episode is numeric but in info, it is of Object data type

anime['episodes'] = pd.to_numeric(anime['episodes'], errors='coerce')
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  11954 non-null  float64
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 672.5+ KB


In [5]:
anime.describe()

,anime_id,episodes,rating,members
count,12294.000000,11954.000000,12064.000000,1.229400e+04
mean,14058.221653,12.382550,6.473902,1.807134e+04
std,11455.294701,46.865352,1.026746,5.482068e+04
min,1.000000,1.000000,1.670000,5.000000e+00
25%,3484.250000,1.000000,5.880000,2.250000e+02
50%,10260.500000,2.000000,6.570000,1.550000e+03
75%,24794.500000,12.000000,7.180000,9.437000e+03
max,34527.000000,1818.000000,10.000000,1.013917e+06


In [6]:
# Lets find null values

anime.isna().sum()

anime_id      0
name          0
genre        62
type         25
episodes    340
rating      230
members       0
dtype: int64

In [7]:
# Lets fill null values

avg_rating = anime["rating"].mean()
print("avg_rating",avg_rating)

print('-'*20)


avg_episode = anime["episodes"].mean()
print("avg_episode",avg_episode)

print('-'*20)

mode_genre = anime["genre"].mode()[0]
print("mode_genre",mode_genre)
     
print('-'*20)

mode_type = anime["type"].mode()[0]
print("mode_type",mode_type)

avg_rating 6.473901690981432
--------------------
avg_episode 12.382549774134182
--------------------
mode_genre Hentai
--------------------
mode_type TV


In [8]:
anime.fillna({
    'genre' : mode_genre,
    'rating' : avg_rating,
    'type' : mode_type,
    'episodes' : avg_episode
},inplace=True)

In [9]:
anime.isna().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [10]:
anime.shape

(12294, 7)

In [11]:
anime['name'].nunique()

12292

In [12]:
anime['genre'].nunique()

3264

In [13]:
anime['type'].nunique()

6

# Feature Extraction:

Labelling Categorical Values

In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
le = LabelEncoder()
anime['type'] = le.fit_transform(anime['type'])
anime.head(2)

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",0,1.0,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",5,64.0,9.26,793665


Features to use:
- Genre (very important for similarity)
- Rating
- Episodes
- Members (popularity)

In [16]:
#Convert Genre to Numerical

from sklearn.preprocessing import MultiLabelBinarizer

anime['genre'] = anime['genre'].apply(lambda x: x.split(', '))

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(anime['genre'])

anime.head(1)

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"[Drama, Romance, School, Supernatural]",0,1.0,9.37,200630


In [17]:
# Normalize

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

numerical_features = scaler.fit_transform(anime[['rating', 'episodes', 'members']])

anime.head(1)

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"[Drama, Romance, School, Supernatural]",0,1.0,9.37,200630


In [28]:
# Combine features

import numpy as np

feature_matrix = np.hstack((genre_matrix, numerical_features))
feature_matrix

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        9.24369748e-01, 0.00000000e+00, 1.97872202e-01],
       [1.00000000e+00, 1.00000000e+00, 0.00000000e+00, ...,
        9.11164466e-01, 3.46725371e-02, 7.82770102e-01],
       [1.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        9.09963986e-01, 2.75178866e-02, 1.12689267e-01],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        3.85354142e-01, 1.65107320e-03, 2.11063682e-04],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        3.97358944e-01, 0.00000000e+00, 1.67667411e-04],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        4.54981993e-01, 0.00000000e+00, 1.35120208e-04]])

# Item-Item Similarity

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(feature_matrix)
cosine_sim

array([[1.        , 0.31068191, 0.13938585, ..., 0.15027137, 0.15431875,
        0.17306034],
       [0.31068191, 1.        , 0.35863381, ..., 0.11282056, 0.11583098,
        0.12988786],
       [0.13938585, 0.35863381, 1.        , ..., 0.11687054, 0.12000412,
        0.1345798 ],
       ...,
       [0.15027137, 0.11282056, 0.11687054, ..., 1.        , 0.99994463,
        0.99824866],
       [0.15431875, 0.11583098, 0.12000412, ..., 0.99994463, 1.        ,
        0.99881138],
       [0.17306034, 0.12988786, 0.1345798 , ..., 0.99824866, 0.99881138,
        1.        ]])

In [20]:
# Fill diagonal

np.fill_diagonal(cosine_sim , 0)

# Recommendation System:

In [21]:
def recommend_anime(title, df, cosine_sim, top_n):
    indices = pd.Series(anime.index, index=anime['name']).drop_duplicates()
    
    if title not in indices:
        return "Anime not found"
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip itself
    sim_scores = sim_scores[1:top_n+1]
    
    anime_indices = [i[0] for i in sim_scores]
    
    return anime['name'].iloc[anime_indices]

In [22]:
recommend_anime("Kimi no Na wa.", anime, cosine_sim, 5)

6394                         Wind: A Breath of Heart (TV)
1111                Aura: Maryuuin Kouga Saigo no Tatakai
504     Clannad: After Story - Mou Hitotsu no Sekai, K...
208                         Kokoro ga Sakebitagatterunda.
1201                       Angel Beats!: Another Epilogue
Name: name, dtype: object

In [23]:
recommend_anime("Aura: Maryuuin Kouga Saigo no Tatakai", anime, cosine_sim, 7)

613              Little Busters!: EX
320     Kokoro Connect: Michi Random
518                   Kokoro Connect
2724     Otome wa Boku ni Koishiteru
4219     Rokujouma no Shinryakusha!?
2078                     Kotoura-san
5346        Ajimu: Kaigan Monogatari
Name: name, dtype: object

In [24]:
def recommend_with_threshold(title, threshold):
    idx = anime[anime['name'] == title].index[0]
    scores = list(enumerate(cosine_sim[idx]))
    
    filtered = [i for i in scores if i[1] > threshold]
    filtered = sorted(filtered, key=lambda x: x[1], reverse=True)
    
    return anime['name'].iloc[[i[0] for i in filtered[1:]]]

In [25]:
recommend_with_threshold("Wind: A Breath of Heart (TV)", 0.9)

0                              Kimi no Na wa.
1111    Aura: Maryuuin Kouga Saigo no Tatakai
Name: name, dtype: object

In [26]:
recommend_with_threshold("Wind: A Breath of Heart (TV)", 0.8)

0                                          Kimi no Na wa.
1111                Aura: Maryuuin Kouga Saigo no Tatakai
6160                  Tokimeki Memorial: Forever With You
6156                                      School Days ONA
5233                                   To Heart 2 Special
5127                                   Venus Versus Virus
5031                                       Mizuiro (2003)
4514                                         Touka Gettan
3914                           Myself ; Yourself Specials
3908     Koi to Senkyo to Chocolate: Ikenai Hazuki-sensei
3530         Otome wa Boku ni Koishiteru: Futari no Elder
3297                   Koi to Senkyo to Chocolate Special
1959                                            Air Movie
2300                           Koi to Senkyo to Chocolate
1494                                             Harmonie
1436                   &quot;Bungaku Shoujo&quot; Memoire
1631                                  Kimikiss Pure Rouge
1907          

In [27]:
recommend_with_threshold("Wind: A Breath of Heart (TV)", 0.9)

0                              Kimi no Na wa.
1111    Aura: Maryuuin Kouga Saigo no Tatakai
Name: name, dtype: object

## Areas of Improvement

- Add user rating matrix (collaborative filtering)
- Use weighted features (e.g., genre > rating)
- Removing noise (very low-rated anime)
- Try hybrid models
- Cold Start: Add content-based features for new users/items.
- Scalability : Precompute similarities or switch to model-based CF.

# Interview Questions

1. Can you explain the difference between user-based and item-based collaborative filtering?

- User-based collaborative filtering finds similar users, while item-based collaborative filtering finds similar items. In practice, user-based methods recommend items liked by “taste twins,” whereas item-based methods recommend items that are most similar to what the user has already consumed

User-Based Collaborative Filtering (UBCF):
- Find users with similar preferences and recommend items they liked.
- Example,If User A and User B both liked Naruto and Bleach, and User B also liked One Piece, recommend One Piece to User A.
- Requires comparing users against each other (user-user similarity).
- Less scalable for large datasets because user preferences change frequently.
- Use Cases : Social platforms, personalized recommendations based on peer groups.

Item-Based Collaborative Filtering (IBCF):
- Find items similar to those the user has already interacted with.
- If a user liked Naruto, recommend Bleach because many users who liked Naruto also liked Bleach
- Requires comparing items against each other (item-item similarity)
- More scalable since item similarities are relatively stable over time.
- Use cases : E-commerce, streaming services where item similarity is strong (e.g., movies, products).


2. What is Collaborative Filtering?
- Collaborative filtering is a recommendation technique that uses user behavior (ratings, interactions) to suggest items.

How it works:
- Build a user-item matrix

Find similarities:
- Between users OR Between items
- Predict preferences

Types:
- User-based : Finds users with similar tastes and recommends items they liked.
- Item-based : Finds items similar to those the user liked and recommends them.





Real world applications:

- Netflix: Recommends shows based on what similar viewers watched.

- Amazon: Suggests products often bought together by similar customers.

- Spotify: Recommends songs based on listening patterns of similar users.